In [43]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.formula.api as smf
from sklearn.metrics import accuracy_score, roc_auc_score
from scipy.stats import ks_2samp

df = pd.read_feather(r'./Environment\workspace\credit_risk_project\exercicios\credit_scoring.ftr')
df.head()

# Tarefa II

Neste projeto, estamos construindo um credit scoring para cartão de crédito, em um desenho amostral com 15 safras, e utilizando 12 meses de performance.

Carregue a base de dados ```credit_scoring.ftr```.

In [44]:
# %% 
# Pre-processamento e Divisão OOT
df['data_ref'] = pd.to_datetime(df['data_ref'])
df['mau'] = df['mau'].astype(int)
df['ano_mes'] = df['data_ref'].dt.to_period('M')

meses_ordenados = df['ano_mes'].sort_values().unique()
ultimos_3_meses = meses_ordenados[-3:]

df_dev = df[~df['ano_mes'].isin(ultimos_3_meses)].copy()
df_oot = df[df['ano_mes'].isin(ultimos_3_meses)].copy()

print(f"Desenvolvimento: {df_dev.shape} | OOT: {df_oot.shape}")
print(f"Período DEV: {df_dev['data_ref'].min().date()} a {df_dev['data_ref'].max().date()}")
print(f"Período OOT: {df_oot['data_ref'].min().date()} a {df_oot['data_ref'].max().date()}")

## Amostragem

Separe os três últimos meses como safras de validação *out of time* (oot).

Variáveis:<br>
Considere que a variável ```data_ref``` não é uma variável explicativa, é somente uma variável indicadora da safra, e não deve ser utilizada na modelagem. A variávei ```index``` é um identificador do cliente, e também não deve ser utilizada como covariável (variável explicativa). As restantes podem ser utilizadas para prever a inadimplência, incluindo a renda.


## Descritiva básica univariada

- Descreva a base quanto ao número de linhas, número de linhas para cada mês em ```data_ref```.
- Faça uma descritiva básica univariada de cada variável. Considere as naturezas diferentes: qualitativas e quantitativas.

In [45]:
print('--- DIMENSÕES DA BASE (DEV) ---')
print(f'Linhas: {df_dev.shape[0]} | Colunas: {df_dev.shape[1]}\n')

print('--- LINHAS POR MÊS ---')
print(df['data_ref'].value_counts().sort_index())
print('\n')

print('--- DESCRITIVA: VARIÁVEIS QUANTITATIVAS ---')
display(df_dev.describe())

print('--- DESCRITIVA: VARIÁVEIS QUALITATIVAS ---')
display(df_dev.describe(include=['object', 'category', 'bool']))

## Descritiva bivariada

Faça uma análise descritiva bivariada de cada variável

In [46]:
sns.set_theme(style='whitegrid')
cat_cols = df_dev.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
if 'mau' in cat_cols: cat_cols.remove('mau')
if 'data_ref' in cat_cols: cat_cols.remove('data_ref')

for col in cat_cols:
    plt.figure(figsize=(10, 5))
    sns.pointplot(data=df_dev, x=col, y='mau')
    plt.title(f'Taxa de Mau por {col}')
    plt.xticks(rotation=45)
    plt.show()

## Desenvolvimento do modelo

Desenvolva um modelo de *credit scoring* através de uma regressão logística.

- Trate valores missings e outliers
- Trate 'zeros estruturais'
- Faça agrupamentos de categorias conforme vimos em aula
- Proponha uma equação preditiva para 'mau'
- Caso hajam categorias não significantes, justifique

In [47]:
df_dev_sm = df_dev.copy()
df_dev_sm['tempo_emprego'] = df_dev_sm['tempo_emprego'].fillna(-1)

# Incluindo variáveis e a Renda
formula = 'mau ~ sexo + posse_de_veiculo + posse_de_imovel + qtd_filhos + tipo_renda + educacao + estado_civil + tipo_residencia + idade + tempo_emprego + qt_pessoas_residencia + renda'
model_log = smf.logit(formula, data=df_dev_sm).fit()
print(model_log.summary())

## Avaliação do modelo

Avalie o poder discriminante do modelo pelo menos avaliando acurácia, KS e Gini.

Avalie estas métricas nas bases de desenvolvimento e *out of time*.

In [48]:
def avaliar(df, model, label):
    df = df.copy()
    # Garantir que 'mau' seja numérico para o cálculo das métricas
    df['mau'] = df['mau'].astype(int)
    # Pre-process for evaluation (same logic as training)
    df['tempo_emprego'] = df['tempo_emprego'].fillna(-1)
    prob = model.predict(df)
    auc = roc_auc_score(df['mau'], prob)
    gini = 2 * auc - 1
    ks = ks_2samp(prob[df['mau'] == 1], prob[df['mau'] == 0]).statistic
    print(f'{label}: Gini = {gini:.4f}, KS = {ks:.4f}')

avaliar(df_dev, model_log, 'DEV')
avaliar(df_oot, model_log, 'OOT')

## Criar um pipeline utilizando o sklearn pipeline 

In [49]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin

class OutlierRemover(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.99):
        self.threshold = threshold
        self.limits = None
    def fit(self, X, y=None):
        if hasattr(X, 'quantile'): 
            self.limits = X.quantile(self.threshold)
        else: 
            self.limits = np.percentile(X, self.threshold * 100, axis=0)
        return self
    def transform(self, X):
        X_copy = X.copy()
        if hasattr(X_copy, 'columns'):
            for i, col in enumerate(X_copy.columns):
                limit = self.limits[col]
                X_copy[col] = np.where(X_copy[col] > limit, limit, X_copy[col])
        else:
            for i in range(X_copy.shape[1]):
                limit = self.limits[i]
                X_copy[:, i] = np.where(X_copy[:, i] > limit, limit, X_copy[:, i])
        return X_copy

cols_to_drop = ['mau', 'data_ref', 'ano_mes', 'index']
X_train = df_dev.drop(columns=[c for c in cols_to_drop if c in df_dev.columns], errors='ignore')
y_train = df_dev['mau'].astype(int)

num_features = X_train.select_dtypes(include=['number']).columns.tolist()
cat_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

pipe = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('outliers', OutlierRemover()),
            ('scaler', StandardScaler())
        ]), num_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_features)
    ])),
    ('pca', PCA(n_components=5)),
    ('classifier', LogisticRegression())
])

pipe.fit(X_train, y_train)
print('Pipeline treinado com sucesso.')

## Pré processamento

### Substituição de nulos (nans)

Existe nulos na base? é dado numérico ou categórico? qual o valor de substituição? média? valor mais frequente? etc

In [50]:
df_proc = df_dev.copy()
print(f'Nulos antes:\n{df_proc.isna().sum()}')

# Substituindo nulos em tempo_emprego pela mediana
df_proc['tempo_emprego'] = df_proc['tempo_emprego'].fillna(df_proc['tempo_emprego'].median())

print(f'\nNulos depois:\n{df_proc.isna().sum()}')

### Remoção de outliers

Como identificar outlier? Substituir o outlier por algum valor? Remover a linha?

In [51]:
def tratar_outliers(df, colunas):
    df_copy = df.copy()
    for col in colunas:
        p99 = df_copy[col].quantile(0.99)
        df_copy[col] = np.where(df_copy[col] > p99, p99, df_copy[col])
    return df_copy

num_cols = ['idade', 'tempo_emprego', 'renda', 'qtd_filhos', 'qt_pessoas_residencia']
df_proc = tratar_outliers(df_proc, num_cols)
print('Outliers tratados via clipping (p99).')

### Seleção de variáveis

Qual tipo de técnica? Boruta? Feature importance? 

In [52]:
from sklearn.ensemble import RandomForestClassifier

X_sel = pd.get_dummies(df_proc.drop(columns=['mau', 'data_ref', 'ano_mes', 'index'], errors='ignore'), drop_first=True)
y_sel = df_proc['mau']

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_sel, y_sel)

features = X_sel.columns
importances = rf.feature_importances_
indices = np.argsort(importances)[-10:] # Top 10

plt.title('Top 10 Feature Importances')
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

### Redução de dimensionalidade (PCA)

Aplicar PCA para reduzir a dimensionalidade para 5

In [53]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# O PCA requer dados normalizados e numéricos
X_pca = pd.get_dummies(df_proc.drop(columns=['mau', 'data_ref', 'ano_mes', 'index'], errors='ignore'), drop_first=True)
X_pca_scaled = StandardScaler().fit_transform(X_pca)

pca = PCA(n_components=5)
X_pca_trans = pca.fit_transform(X_pca_scaled)

print(f'Variância explicada acumulada: {pca.explained_variance_ratio_.sum():.4f}')

### Criação de dummies

Aplicar o get_dummies() ou onehotencoder() para transformar colunas catégoricas do dataframe em colunas de 0 e 1. 
- sexo
- posse_de_veiculo
- posse_de_imovel
- tipo_renda
- educacao
- estado_civil
- tipo_residencia

In [54]:
cols_to_dummy = ['sexo', 'posse_de_veiculo', 'posse_de_imovel', 'tipo_renda', 'educacao', 'estado_civil', 'tipo_residencia']
df_dummies = pd.get_dummies(df_proc, columns=cols_to_dummy, drop_first=True)
print(f'Shape após dummies: {df_dummies.shape}')
df_dummies.head()

### Pipeline 

Crie um pipeline contendo essas funções.

preprocessamento()
- substituicao de nulos
- remoção outliers
- PCA
- Criação de dummy de pelo menos 1 variável (posse_de_veiculo)

In [55]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

X = df_dev.drop(columns=['mau', 'data_ref', 'ano_mes', 'index'], errors='ignore')
y = df_dev['mau'].astype(int)

num_features = X.select_dtypes(include=['number']).columns.tolist()
cat_features = X.select_dtypes(exclude=['number']).columns.tolist()

pipe_final = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('outliers', OutlierRemover()), # Classe definida anteriormente
            ('scaler', StandardScaler())
        ]), num_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_features)
    ])),
    ('pca', PCA(n_components=5)),
    ('classifier', LogisticRegression())
])

print('Pipeline estruturado.')

### Treinar um modelo de regressão logistica com o resultado

In [56]:
pipe_final.fit(X, y)
print('Modelo treinado via Pipeline.')

### Salvar o pickle file do modelo treinado

In [57]:
import pickle
with open('model_final.pkl', 'wb') as f:
    pickle.dump(pipe_final, f)
print('Pickle salvo: model_final.pkl')

In [58]:
# NOTA: O Pycaret não suporta Python 3.12 (versão atual do ambiente).
# Como alternativa, utilizaremos a biblioteca 'lightgbm' diretamente para treinar o modelo solicitado.

import lightgbm as lgb
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.metrics import roc_curve, auc
from sklearn.impute import SimpleImputer

# Preparação dos dados
df_lgb_copy = df_dev.copy()
# Tratamento de nulos em tempo_emprego (SMOTE não aceita NaNs)
df_lgb_copy['tempo_emprego'] = df_lgb_copy['tempo_emprego'].fillna(df_lgb_copy['tempo_emprego'].median())

X_lgb = pd.get_dummies(df_lgb_copy.drop(columns=['data_ref', 'ano_mes', 'index'], errors='ignore'), drop_first=True)
y_lgb = df_lgb_copy['mau'].astype(int)

X_train_l, X_val_l, y_train_l, y_val_l = train_test_split(X_lgb, y_lgb, test_size=0.2, random_state=123)

# Tratamento de desbalanceamento (equivalente ao fix_imbalance do Pycaret)
smote = SMOTE(random_state=123)
X_res, y_res = smote.fit_resample(X_train_l, y_train_l)

# Treinamento do LightGBM
clf_lgb = lgb.LGBMClassifier(random_state=123, verbose=-1)
clf_lgb.fit(X_res, y_res)

# Avaliação Visual (AUC)
prob_lgb = clf_lgb.predict_proba(X_val_l)[:, 1]
fpr, tpr, thresholds = roc_curve(y_val_l, prob_lgb)
plt.plot(fpr, tpr, label=f'AUC = {auc(fpr, tpr):.4f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('LightGBM - Curva ROC (Validação)')
plt.legend()
plt.show()

# Salvando o modelo
import joblib
joblib.dump(clf_lgb, 'lightgbm_model.pkl')
print('Modelo LightGBM treinado e salvo como lightgbm_model.pkl')

# Projeto Final

1. Subir no GITHUB todos os jupyter notebooks/códigos que você desenvolveu nesse ultimo módulo
1. Gerar um arquivo python (.py) com todas as funções necessárias para rodar no streamlit a escoragem do arquivo de treino
    - Criar um .py
    - Criar um carregador de csv no streamlit 
    - Subir um csv no streamlit 
    - Criar um pipeline de pré processamento dos dados
    - Utilizar o modelo treinado para escorar a base 
        - nome_arquivo = 'model_final.pkl'
1. Gravar um vídeo da tela do streamlit em funcionamento (usando o próprio streamlit (temos aula disso) ou qlqr outra forma de gravação).
1. Subir no Github o vídeo de funcionamento da ferramenta como README.md.
1. Subir no Github os códigos desenvolvidos. 
1. Enviar links do github para o tutor corrigir.